# LC 236 — Lowest Common Ancestor of a Binary Tree
**Day 36 | DFS on Binary Trees | Medium**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Post-order DFS returns the
node itself when found. If a node sees <em>both</em> left
and right return non-None, it is the split point — the LCA.
If only one side returns non-None, bubble that up.
</div>

## Official Problem Statement

Given a binary tree, find the **lowest common ancestor (LCA)**
of two given nodes in the tree.

The LCA of two nodes `p` and `q` is defined as the lowest node
in the tree that has both `p` and `q` as descendants (a node
is allowed to be a descendant of itself).

**Constraints:**
- The number of nodes is in the range `[2, 10^5]`.
- `-10^9 <= Node.val <= 10^9`
- All `Node.val` are **unique**.
- `p != q`
- Both `p` and `q` exist in the tree.

## What This Is Actually Asking

Find the deepest node that is an ancestor of both p and q.
A node counts as its own ancestor, so if p is an ancestor of
q, then p is the LCA.
The tree is not a BST — we cannot use value comparisons to
navigate; we must search the whole tree.
Both nodes are guaranteed to exist, so we do not need to
handle the case where one is missing.
The answer is always unique because all node values are unique.

## Walk Through an Example by Hand

Tree: `[3,5,1,6,2,0,8,None,None,7,4]`
Find LCA of p=5, q=1

```
         3
        / \
       5   1
      / \ / \
     6  2 0  8
       / \
      7   4
```

1. DFS reaches node 5 — matches p → return node 5 upward.
2. DFS reaches node 1 — matches q → return node 1 upward.
3. At root 3: left=5 (non-None), right=1 (non-None).
4. Both sides non-None → root 3 is the LCA. **Answer: 3**

Find LCA of p=5, q=4:
1. At node 5: root==p → return node 5 immediately.
2. Node 4 is in subtree of 5, but we already returned 5.
3. Right side of root 3 returns None (1's subtree has no
   p or q). Left side returned 5.
4. Only left side non-None → bubble 5 up. **Answer: 5**

## The Picture

Post-order: children report up; LCA is the split point.

```
         3        <- left=5, right=1 -> BOTH -> LCA=3
        / \
       5   1      <- 5==p -> return 5 | 1==q -> return 1
      / \ / \
     6  2 0  8    <- return None (not p or q)
       / \
      7   4

Signal flow for LCA(p=5, q=1):
  6 -> None
  7 -> None
  4 -> None
  2 -> None (2 is not p or q, neither child found)
  5 -> return self (5 == p, stop descending)
  0 -> None
  8 -> None
  1 -> return self (1 == q)
  3: left=5, right=1 -> BOTH non-None -> return 3 *

Key rule:
  left AND right non-None  -> current node is LCA
  only left non-None       -> bubble left up
  only right non-None      -> bubble right up
  both None                -> return None
```

## When To Use This Pattern

- When searching for a node **by identity** (not value range),
  think full DFS — not BST navigation.
- When the answer is determined by **signals from both
  subtrees**, think post-order DFS with return values.
- When a node can **be its own ancestor**, handle the base
  case as: `if root == p or root == q: return root`.
- When you need to **bubble a finding upward** through many
  levels, returning the node itself from DFS is cleaner
  than using a global variable.
- When both targets are guaranteed present, return early on
  match without exploring the node's subtrees.

## The Approach

Use recursive post-order DFS. If the current node is None,
or equals p or q, return it immediately. Recurse into both
children. If both return non-None, the current node is the
split point — return it as the LCA. Otherwise return whichever
side is non-None, propagating the found node upward.

In [ ]:
from collections import deque
from typing import Optional


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def make_tree(vals):
    """BFS builder: construct TreeNode tree from level-order list."""
    if not vals:
        return None
    root = TreeNode(vals[0])
    q = deque([root])
    i = 1
    while q and i < len(vals):
        node = q.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i])
            q.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i])
            q.append(node.right)
        i += 1
    return root


def find_node(root, val):
    """Helper: find and return node with given val via DFS."""
    if not root:
        return None
    if root.val == val:
        return root
    return (
        find_node(root.left, val)
        or find_node(root.right, val)
    )

In [ ]:
def test_harness(func):
    """Run test cases for lowestCommonAncestor."""
    cases = [
        # (tree_vals, p_val, q_val, expected_lca_val, label)
        (
            [3,5,1,6,2,0,8,None,None,7,4],
            5, 1, 3,
            "LC example 1 — p=5, q=1, LCA=3"
        ),
        (
            [3,5,1,6,2,0,8,None,None,7,4],
            5, 4, 5,
            "LC example 2 — p=5, q=4, LCA=5 (ancestor=self)"
        ),
        (
            [1, 2],
            1, 2, 1,
            "Edge: root is LCA of root and child"
        ),
        (
            [3,5,1,6,2,0,8,None,None,7,4],
            7, 4, 2,
            "Siblings under node 2"
        ),
        (
            [3,5,1,6,2,0,8,None,None,7,4],
            6, 4, 5,
            "p=6 and q=4 both under node 5"
        ),
    ]
    passed = 0
    for vals, p_val, q_val, exp_val, label in cases:
        root = make_tree(vals)
        p = find_node(root, p_val)
        q = find_node(root, q_val)
        result = func(root, p, q)
        got_val = result.val if result else None
        status = "PASSED" if got_val == exp_val else "FAILED"
        if status == "PASSED":
            passed += 1
        print(f"{status} | {label} "
              f"| expected={exp_val} got={got_val}")
    print(f"\n{passed}/{len(cases)} tests passed.")


print("test_harness defined — run after implementing solution.")

In [ ]:
def lowestCommonAncestor(
    root: TreeNode,
    p: TreeNode,
    q: TreeNode
) -> TreeNode:
    """
    Return the lowest common ancestor of nodes p and q.

    Strategy: Recursive post-order DFS.
      - Base: None or root==p or root==q -> return root.
      - Recurse left and right.
      - Both non-None: current node is the LCA.
      - Else: return whichever side is non-None.

    Args:
        root: Root of the binary tree.
        p:    First target node.
        q:    Second target node.

    Returns:
        The LCA TreeNode.

    Time:  O(n) — may visit all nodes.
    Space: O(h) — call stack height.
    """
    # Debug: trace current node
    print(f"  visiting: {root.val if root else None} "
          f"| p={p.val} q={q.val}")

    # Base case: None or target node found
    # TODO: if not root or root == p or root == q: return root

    # Recurse into both subtrees
    # TODO: left  = lowestCommonAncestor(root.left, p, q)
    # TODO: right = lowestCommonAncestor(root.right, p, q)

    # Debug: show what each subtree returned
    # lv = left.val  if left  else None
    # rv = right.val if right else None
    # print(f"  node={root.val} left_ret={lv} right_ret={rv}")

    # Both sides found something: current node is the split
    # TODO: if left and right: return root

    # Bubble up whichever side found something
    # TODO: return left or right

    pass

In [ ]:
# Uncomment and run when solution is ready
# test_harness(lowestCommonAncestor)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| Store all paths, compare | O(n) | O(n) | Two root-to-node paths |
| Recursive DFS (optimal) | O(n) | O(h) | Single traversal |
| Parent pointer map + sets | O(n) | O(n) | Two passes; more code |

**n** = nodes, **h** = height (O(log n) balanced, O(n) skewed).

## Real World Connection

At **Citi**, org chart queries like "find the shared reporting
manager for employee A and employee B" are exactly the LCA
problem — used in access control and escalation routing.
On **AWS Organizations**, finding the lowest common
organizational unit (OU) governing two accounts is an LCA
query that determines which SCP policies apply to both.
As a **data engineer**, lineage graphs in tools like dbt or
Apache Atlas are DAGs; finding the common upstream dependency
between two tables is conceptually LCA on a tree.
Version control systems use LCA to find the merge base between
two branches — the foundation of git merge and rebase.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra